# Извлечение ключевых слов из аннотации к статье

Файлы:
*  Adaskina_IMS_2015_KW_ru.txt - ключевые слова
*  Adaskina_IMS_2015_Abstract_ru.txt - аннотация
*  Adaskina_IMS_2015_ru.txt - сама статья

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install pymorphy2

## RAKE

In [ ]:
import pymorphy2
import string
import operator
import nltk
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

Подгрузим стоп-слова из разных файлов

In [ ]:
# пунктуация
with open("/content/drive/My Drive/Семантические анализаторы/stopwords/punct.txt", 'r', encoding='utf-8') as f1:
  punct = f1.read()
punct_list = nltk.word_tokenize(punct)

# ещё какие-то стоп-слова
with open("/content/drive/My Drive/Семантические анализаторы/stopwords/stop.txt", 'r', encoding='utf-8') as f2:
  stop = f2.read()

# местоимения
with open("/content/drive/My Drive/Семантические анализаторы/stopwords/pronouns.txt", 'r', encoding='utf-8') as f3:
  pronouns = f3.read()

# предлоги
with open("/content/drive/My Drive/Семантические анализаторы/stopwords/preps.txt", 'r', encoding='utf-8') as f4:
  preps = f4.read()

# авторские стоп-слова!
with open("/content/drive/My Drive/Семантические анализаторы/stopwords/userstop.txt", 'r', encoding='utf-8') as f5:
  mystop = f5.read()

# общий список стоп-слов
stop_list = nltk.word_tokenize(stop) + nltk.word_tokenize(pronouns) + nltk.word_tokenize(preps) + nltk.word_tokenize(mystop)
len(stop_list)

1134

Определим ряд функций для проверки слов на вшивость.

In [ ]:
# является ли токен знаком препинания
def isPunct(word):
  return len(word) == 1 and (word in string.punctuation or word in punct_list)

# является ли токен целым или вещественным числом
def isNumeric(word):
  try:
    float(word) if '.' in word else int(word)
    return True
  except ValueError:
    return False

# является ли токен однобуквенным сокращением (инициалом)
def isInitial(word):
  return len(word) == 2 and word[1] == "."

Определим непосредственно сам класс для выделения ключевых слов и выражений.

In [ ]:
class RakeKeywordExtractor:

  def __init__(self):
    self.stopwords = set(nltk.corpus.stopwords.words()) # определяем список стоп-слов - загружаем его из nltk
    self.top_fraction = 1 # consider top third candidate keywords by score

  """Метод для генерации кандидатов в ключевые слова, на вход получает список предложений"""
  def _generate_candidate_keywords(self, sentences):
    phrase_list = []  # список ключевых слов и фраз (каждый элемент списка phrase_list - тоже список!)
    for sentence in sentences:
      words = map(lambda x: "|" if x in self.stopwords or x in stop_list or isNumeric(x) or isInitial(x) else x,
        nltk.word_tokenize(sentence.lower())) # функция map используется для применения некоторой функции ко всем элементам итерируемого объекта
        # в данном случае мы проходим по всем токенам предложения, после чего превращаем слово в палку (|), если это стоп-слово, число или видимо однобуквенное сокращение
        # таким образом мы находим границы между кандидатами в ключевые слова!
      phrase = []
      for word in words: # проходим по словам данного предложения
        if word == "|" or isPunct(word): # если данный токен - палка (граница) или знак препинания (тоже граница)
          if len(phrase) > 0: # если в списке для данного ключевого выражения уже есть какие-то слова
            phrase_list.append(phrase) # добавляем это ключевое выражение (в виде списка слов) в общему списку ключевых выражений
            phrase = [] # обнуляем список для текущего ключевого выражения
        else: # иначе, то есть если у нас пока нет никаких слов
          phrase.append(word) # палка или знак препинания начинает новое ключевое выражение
    return phrase_list # возвращаем список ключевых выражений

  """Функция для вычисления оценок для слов"""
  def _calculate_word_scores(self, phrase_list):
    word_freq = nltk.FreqDist() # распределение частот: сколько раз тот или иной исход зафиксирован при выполнении данного эксперимента?
    word_degree = nltk.FreqDist() # видимо, это словарь
    for phrase in phrase_list: # в цикле перебираем ключевые выражения
      #degree = len(filter(lambda x: not isNumeric(x), phrase)) - 1
      degree = len(list(filter(lambda x: not isNumeric(x), phrase))) - 1 # filter СОХРАНЯЕТ те элементы списка, для которых функция (в данном случае - это lambda) вернула TRUE
      # то есть из списка для данного ключевого выражения мы выкидываем числа (not isNumeric(x) = True для всего, что не число)
      # degree - длина словосочетания, содержащего данное слово
      for word in phrase:
        word_freq[word] += 1 # считаем частоту слова (во всём корпусе, если я правильно понимаю)
        word_degree[word] += degree # считаем степень слова (тоже для всего корпуса)
    for word in word_freq.keys():
      word_degree[word] = word_degree[word] + word_freq[word] # пересчитываем значение degree для каждого слова: оно равняется частоте слова в корпусе + сумме длин словосочетаний с этим словом
    # word score = deg(w) / freq(w)
    word_scores = {} # финальный словарь с оценками для слов
    for word in word_freq.keys():
      word_scores[word] = word_degree[word] / word_freq[word] # оценка равна степень слова / частоту слова
    return word_scores

  """Функция для подсчёта оценок для ключевых выражений (уже не для слов!)"""
  def _calculate_phrase_scores(self, phrase_list, word_scores):
    phrase_scores = {}
    for phrase in phrase_list:
      phrase_score = 0
      for word in phrase:
        phrase_score += word_scores[word] # оценка ключевого выражения = сумма оценок входящих в него слов
      phrase_scores[" ".join(phrase)] = phrase_score
    return phrase_scores

  """Общая функция для извлечения ключевых слов"""
  def extract(self, text, incl_scores=False):
    sentences = nltk.sent_tokenize(text) # делим текст на предложения
    phrase_list = self._generate_candidate_keywords(sentences) # генерируем кандидатов в ключевые слова (на выходе - список ключевых выражений вида [[w1, w2...], [], ... ,[]])
    word_scores = self._calculate_word_scores(phrase_list) # считаем оценки для всех слов корпуса (на выходе - словарь вида {слово: оценка})
    phrase_scores = self._calculate_phrase_scores(
      phrase_list, word_scores) # считаем оценки для ключевых выражений (на выходе - словарь вида {ключевое выражение в виде строки: оценка})
    #sorted_phrase_scores = sorted(phrase_scores.iteritems(),
    sorted_phrase_scores = sorted(phrase_scores.items(),
      key=operator.itemgetter(1), reverse=True) # сортируем словарь с ключевыми выражениями по убыванию значений оценок
      # key=функция - по результату выполнения какой функции сортировать
      # operator.itemgetter(1) - будем возвращать второй элемент каждого кортежа, то есть оценку для данного ключенвого выражения
    n_phrases = len(sorted_phrase_scores) # сколько всего ключевых выражений?
    if incl_scores: # по умолчанию incl_scores = False
      return sorted_phrase_scores[0:int(n_phrases/self.top_fraction)]
    else:
      return map(lambda x: x[0],
        sorted_phrase_scores[0:int(n_phrases/self.top_fraction)])

In [ ]:
# создадим объект класса
rake = RakeKeywordExtractor()

In [ ]:
# подгрузим наш текст
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_Abstract_rus.txt', 'r', encoding='utf-8') as f:
    txt = f.read()

In [ ]:
# Это была аннотация, а есть ещё вообще-то полный текст
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
    full_text = f.read()

In [ ]:
# извлечём ключевые слова, используя метод extract() класса RakeKeywordExtractor
keywords = rake.extract(txt, incl_scores=True)

In [ ]:
keywords = [k[0] for k in keywords]
keywords

['работа посвящена опыту применения полуавтоматического метода',
 'использовании предлагаемого метода достаточно высокая полнота',
 'проведена ручная разметка корпуса',
 'итеративной модели классификации',
 'частичным привлечением учителя',
 'оценки качества классификации',
 'рамках нашего эксперимента',
 'проверили эффективность использования',
 'сравнили показатели эффективности',
 'размера шага просмотра',
 'значительно снижает трудозатраты',
 'снижения трудозатрат эксперта',
 'документов корпуса',
 'стороны эксперта',
 'настройке классификатора',
 'ключевых словах',
 'метод построен',
 'основе обучения',
 'видов параметров',
 'синтаксических связей',
 'оцениваемых экспертом',
 'каждой итерации',
 'эксперимент показал',
 'количества документов',
 'документов',
 'основанного',
 'лемм',
 'биграмм',
 'комбинаций',
 'зависимости',
 'т.е',
 '0,91',
 'достигнута',
 'просмотре']

Сравним этот набор с ключевыми словами, выделенными автором статьи.

*  классификация;
*  машинное обучение;
*  обучение с частичным привлечением учителя.

In [ ]:
# правильные keywords
gold = ['классификация', 'машинное обучение', 'обучение с частичным подкреплением']

In [ ]:
tp = 0
for w in gold:
  if w in keywords:
    tp += 1
tp

0

In [ ]:
# сколько КС алгоритм выделил из аннотации?
len(keywords)

34

Ну вот, ничего не выделил:(

А что с полным текстом статьи?

In [ ]:
keywords_full = [kw[0] for kw in rake.extract(full_text, incl_scores=True)]
keywords_full

['изначально заданными классами необходимо вручную разметить небольшой фрагмент выборки документов',
 'нулевом шаге эксперт самостоятельно размечает небольшое подмножество исходной выборки',
 'алгоритм позволил достичь приемлемого качества анализа небольшого специализированного корпуса',
 'несколько итераций подавляющее большинство релевантных документов окажутся вверху сортированной выборки',
 'использование ранжирования облегчает задачу экспертной верификации автоматической классификации',
 'большим числом классов возможно последовательное применение нашего метода',
 'частичным привлечением учителя позволит существенно снизить трудозатраты эксперта',
 'оптимальный прогон позволяет получить полноту классификации 0,91',
 'получение размеченного тренировочного корпуса достаточного объема затруднено',
 'итеративного алгоритма позволит повысить качество работы алгоритма',
 'подобных алгоритмов является наличие качественных тренировочных данных',
 'далее выявленные автоматически характерис

In [ ]:
tp = 0
for w in gold:
  if w in keywords_full:
    print(w)
    tp += 1
print(tp)

# сколько КС алгоритм выделил из аннотации?
print(len(keywords))

классификация
1
34


## YAKE!

Yet Another Keyword Extractor

repo: https://github.com/LIAAD/yake?tab=readme-ov-file

*  Подход к выделению ключевых слов, основанный на обучении без учителя.
*  Не зависит от корпуса (не требует дообучения).
*  Не зависит от языка и темы документа.
*  Можно применять к одиночным документам.
*  Авторы утверждают, что их моделька вообще всех победала.

In [ ]:
# скачаем библиотеку
!pip install git+https://github.com/LIAAD/yake

  Cloning https://github.com/LIAAD/yake to /tmp/pip-req-build-am7fadus
  Running command git clone --filter=blob:none --quiet https://github.com/LIAAD/yake /tmp/pip-req-build-am7fadus
  Resolved https://github.com/LIAAD/yake to commit 0fa58cceb465162b6bd0cab7ec967edeb907fbcc
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.8 MB/s eta 0:00:00
  Created wheel for yake: filename=yake-0.4.8-py2.py3-none-any.whl size=61995 sha256=d2f95bf1056f67dafa6d9cd71c188b8f133b6090236b0f5a47d26417b53e411b
  Stored in directory: /tmp/pip-ephem-wheel-cache-0aw7uibo/wheels/10/9d/33/6a3358fd876c3d7c6c5c139d1496eb4b1618c7d0e15c375584
Successfully built yake


In [ ]:
import yake
kw_extractor = yake.KeywordExtractor()

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_Abstract_rus.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
keywords = [kw[0] for kw in kw_extractor.extract_keywords(text)]
for kw in keywords:
  print(kw)

Работа посвящена опыту
посвящена опыту применения
опыту применения полуавтоматического
для снижения трудозатрат
применения полуавтоматического метода
снижения трудозатрат эксперта
Работа посвящена
настройке классификатора
основанного на ключевых
ключевых словах
посвящена опыту
опыту применения
применения полуавтоматического
снижения трудозатрат
метода для снижения
полуавтоматического метода для
для снижения
полуавтоматического метода
трудозатрат эксперта
частичным привлечением учителя


In [ ]:
len(keywords)

20

YAKE, в отличие от RAKE, учитывает предлоги. Видно, что алгоритм верно выделил примерно одно КС (частичное привлечение учителя), и то оно не приведено к нормальной форме, плюс потерялось слово "обучение". При этом алгоритм выделил меньше КС, чем RAKE (20 vs 34).

А что с полным текстом статьи?

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
  full_text = f.read()

In [ ]:
keywords_full = [kw[0] for kw in kw_extractor.extract_keywords(full_text)]
for kw in keywords:
  print(kw)

документов
для
классификации
что
выборки
при
параметров
просмотренных документов
для классификации
задачи классификации
связи
при помощи
полноты
классификации текстов
параметров для классификации
Метод
шагом
просмотренных
синтаксические связи
текстов


In [ ]:
print(len(keywords_full))

20


Почему-то из аннотации выделялись неоднословные КВ, а из полного текста - в основном униграммы (странное).

## multi-rake

https://pypi.org/project/multi-rake/#description

In [ ]:
!pip install multi-rake

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 MB 15.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.7/117.7 kB 13.0 MB/s eta 0:00:00
  Created wheel for pycld2: filename=pycld2-0.41-cp310-cp310-linux_x86_64.whl size=9904036 sha256=fe7673f483d980e1785a4da5936836ad5deed2f77ea7a5d50cfc69ade4d6a5e5
  Stored in directory: /root/.cache/pip/wheels/be/81/31/240c89c845e008a93d98542325270007de595bfd356eb0b06c
Successfully built pycld2


In [ ]:
from multi_rake import Rake
rake = Rake()

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_Abstract_rus.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
keywords = [kw[0] for kw in rake.apply(text)]
for kw in keywords:
  print(kw)

итеративной модели классификации
частичным привлечением учителя
оценки качества классификации
рамках нашего эксперимента
сравнили показатели эффективности
размера шага просмотра
значительно снижает трудозатраты
снижения трудозатрат эксперта
стороны эксперта
настройке классификатора
ключевых словах
метод построен
основе обучения
синтаксических связей
оцениваемых экспертом
каждой итерации
эксперимент показал
количества документов
620 документов
основанного
таких
лемм
биграмм
комбинаций
зависимости
достигнута


In [ ]:
len(keywords)

26

Хорошего тут только "частичным привлечением учителя". Всё остальное вообще не очень, в КС даже попало местоимение (*таких*).

А что с полным текстом статьи?

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
  full_text = f.read()

In [ ]:
keywords_full = [kw[0] for kw in rake.apply(full_text)]
for kw in keywords_full:
  print(kw)

несбалансированными классами ю
отдельную сложную задачу
необходимо отсеивать аналитику
имеет четких критериев
большие объемы поступающих
занимающегося подбором лексики
частичным привлечением учителя
исходных частотных термина
краткому описанию работ
каталогизирование новостных статей
использование максимальной энтропии
выявления именованных сущностей
называемая оценка доверия
применялась синтаксическая эвристика
создания лингвистических моделей
решили задействовать имеющийся
получения синтаксической информации
использование синтаксической информации
линейным ядром linearsvc
убыванию данного показателя
небольшом числе итераций
эффективными оказываются параметры
позволяют получить полноту
график наглядно демонстрирует
оказывать существенное влияние
замена такого классификатора
улучшить точность классификатора
посвященной категоризации текстов
желаемого критерия качества
вручную размечены экспертом
первой итерации скрипт
увеличение шага просмотра
нижней границе полноты
высоких уровнях пол

In [ ]:
len(keywords_full)

429

Очень много всего и ничего полезного.

## KeyBERT

KeyBERT: https://github.com/MaartenGr/KeyBERT

Бертовые эмбеддинги слов и н-грамм с помощью метрики косинусного подобия сравниваются с эмбеддингом всего текста: чем выше значение метрики, тем более вероятно, что слово/н-грамма является ключевым выражением.

In [ ]:
!pip install keybert

In [ ]:
from keybert import KeyBERT
kw_model = KeyBERT()

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_Abstract_rus.txt', 'r', encoding='utf-8') as f:
    doc = f.read()

In [ ]:
keywords = [kw[0] for kw in kw_model.extract_keywords(doc)]
keywords

['использования',
 'классификации',
 'применения',
 'эффективности',
 'классификатора']

Правильные keywords:
*  классификация;
*  машинное обучение;
*  обучение с частичным привлечением учителя

Ерунда какая-то получилось, ничего нужного. Но зато не так много, как у предыдущих алгоритмов.

А что с полным текстом статьи?

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
    full_doc = f.read()

In [ ]:
keywords_full = [kw[0] for kw in kw_model.extract_keywords(full_doc)]
keywords_full

['словосочетаний',
 'автоматический',
 'информации',
 'иерархическая',
 'исследований']

Тоже мало и не по делу.

## RuTermExtract

Репозиторий: https://github.com/igor-shevchenko/rutermextract

Библиотека извлекает ключевые слова на основе заранее заданных правил. На данный момент это единственный возможный вариант, поскольку для русского языка не существует открытого синтаксического корпуса, который можно использовать для обучения синтаксических моделей.

In [ ]:
!pip install rutermextract

In [ ]:
from rutermextract import TermExtractor
term_extractor = TermExtractor()

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_Abstract_rus.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
for term in term_extractor(text):
    print(term.normalized, term.count)

частичное привлечение учителя 1
снижение трудозатрат эксперта 1
ручная разметка корпуса 1
размер шага просмотра 1
оценка качества классификации 1
несколько видов параметров 1
итеративная модель классификации 1
эффективность использования 1
сторона эксперта 1
синтаксические связи 1
предлагаемое метод 1
полуавтоматическое метод 1
показатели эффективности 1
оцениваемые эксперт 1
основа обучения 1
опыт применения 1
наше эксперимент 1
наш эксперимент 1
настройка классификатора 1
количество документов 1
ключевые слова 1
каждая итерация 1
документы корпуса 1
высокая полнота 1
620 документов 1
трудозатрата 1
рамки 1
работа 1
просмотр 1
метод 1
леммы 1
комбинации 1
использование 1
зависимость 1
биграмм 1


In [ ]:
len(term_extractor(text))

35

Правильные keywords:
*  классификация;
*  машинное обучение;
*  обучение с частичным привлечением учителя

Тут из хорошего тоже только частичное привлечение учителя.

А что с полным текстом статьи?

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
  full_text = f.read()

In [ ]:
for term in term_extractor(full_text):
    print(term.normalized, term.count)

документы 10
использование 9
биграммы 9
эксперт 8
шаг 8
просмотренные документы 7
работа 7
синтаксические связь 6
классификация 6
классификатор 6
вся выборка 5
помощь 5
обучение 5
метод 5
лемма 5
r 5
br 5
частичное привлечение учителя 4
синтаксические связи 4
спам 4
раздел 4
просмотр 4
получение 4
основа 4
объём 4
выборка 4
целевое класс 3
синтаксическая информация 3
нерелевантные документы 3
наш случай 3
машинное обучение 3
классификация текстов 3
качество параметров 3
каждая итерация 3
итеративный алгоритм 3
эффективность 3
эксперимент 3
число 3
таблица 3
связь 3
классы 3
диапазон 3
графика 3
вероятность 3
алгоритм 3
b 3
отношение полноты классификации 2
коэффициент эффективности метода 2
трудозатрата эксперта 2
тренировочный корпус 2
сортированная выборка 2
семантические классы 2
полнота классификации 2
показывающие зависимость 2
опорные векторы 2
объём выборки 2
новые документы 2
метод обучения 2
малейший шаг 2
ключевые слова 2
задача пополнения 2
достаточное объём 2
вероятность пр

In [ ]:
len(term_extractor(full_text))

436

Здесь уже появляется "классификация", помимо "частичного привлечения учителя". Но "машинного обучения" так и нет, если я не ошибаюсь.

## SpaCY

In [ ]:
!pip install spacy

In [ ]:
# скачиваем модель
!pip install https://github.com/explosion/spacy-models/releases/download/ru_core_news_sm-3.1.0/ru_core_news_sm-3.1.0.tar.gz

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 17.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import spacy
from collections import Counter
from nltk.corpus import stopwords

In [ ]:
# загрузим модель
spacy_model = spacy.load('ru_core_news_sm')

In [ ]:
# подгрузим стоп-слова из nltk и spacy
nltk_stopwords = stopwords.words('russian')
spacy_stopwords = spacy_model.Defaults.stop_words

In [ ]:
def extract_keywords_with_spacy(text):
    keywords = []
    doc = spacy_model(text)
    for token in doc:
        # Тексты уже очищены от знаков препинания и лемматизированы, тем не менее, остались некоторые символы,
        # которые нужно удалить
        if token.text not in spacy_stopwords and token.text not in nltk_stopwords and \
                token.text not in "`'«»...—-":
            if token.pos_ in ('ADJ', 'NOUN', 'VERB'):
                keywords.append(token.text)

    freq_word = Counter(keywords)
    max_freq = Counter(keywords).most_common(1)[0][1]
    for w in freq_word:
        freq_word[w] = round(freq_word[w] / max_freq, 3)

    freq_20 = freq_word.most_common(20)

    return list(freq_20)

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_Abstract_rus.txt', 'r', encoding='utf-8') as f:
  text = f.read()

In [ ]:
keywords = [kw[0] for kw in extract_keywords_with_spacy(text)]
keywords

['документов',
 'метода',
 'эксперта',
 'классификации',
 'корпуса',
 'Работа',
 'посвящена',
 'опыту',
 'применения',
 'полуавтоматического',
 'снижения',
 'трудозатрат',
 'настройке',
 'классификатора',
 'основанного',
 'ключевых',
 'словах',
 'Метод',
 'построен',
 'итеративной']

In [ ]:
len(keywords)

20

Выделяет только отдельные слова, не н-граммы, что в данном случае критично. Таким образом, нормально выделилась только "классификация", и то не в начальной форме.

А что с полным текстом статьи?

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_rus.txt', 'r', encoding='utf-8') as f:
  full_text = f.read()

In [ ]:
keywords_full = [kw[0] for kw in extract_keywords_with_spacy(full_text)]
keywords_full

['документов',
 'классификации',
 'выборки',
 'параметров',
 'полноты',
 'связи',
 'шагом',
 'просмотренных',
 'текстов',
 'классов',
 'эксперта',
 'классификатора',
 'итерации',
 'обучения',
 'классификатор',
 'задачи',
 'алгоритм',
 'принадлежности',
 'качестве',
 'эффективности']

In [ ]:
len(keywords_full)

20

Ровно тот же результат

## Pullenti

Репозиторий: https://github.com/pullenti/pullenti-wrapper?tab=readme-ov-file

Быстрый старт: https://nbviewer.org/github/pullenti/pullenti-wrapper/blob/master/docs.ipynb

In [ ]:
!pip install pullenti_wrapper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 5.2 MB/s eta 0:00:00


In [ ]:
from pullenti_wrapper.processor import Processor, GEO, ORGANIZATION, PERSON, PHONE, ADDRESS
from collections import Counter

pullenti_processor = Processor([PERSON, ORGANIZATION, GEO, PHONE, ADDRESS])

In [ ]:
def extract_keywords_with_pullenti(text):
    keywords = []
    for text in text.read().splitlines():
        try:
            result = pullenti_processor(text) # обрабатываем текст
            for match in result.walk(): # рекурсивно проходим по всем вложенным сущностям
                for key, value in match.referent.slots: # slots - это пары вида ('NAME', 'США')
                    if key not in ('ISRELATIVE', 'SEX', 'NUMBER', 'HIGHER', 'PROFILE',
                                   'ALPHA2', 'POINTER', 'FROM', 'REF', 'ATTRIBUTE'):
                        if 'LASTNAME' in str(match.referent.slots) and 'FIRSTNAME' in str(match.referent.slots): # имя и фамилия есть в slots
                            try:
                                keywords.append(f'{match.referent.firstname} {match.referent.lastname}'.lower())
                            except AttributeError:
                                continue
                        keywords.append(str(value).lower())
        except ValueError:
            continue

    freq_word = Counter(keywords)

    freq_20 = freq_word.most_common(20)

    return list(freq_20)

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_Abstract_rus.txt', 'r', encoding='utf-8') as file:
    keywords = extract_keywords_with_pullenti(file)
keywords

[]

Pullenti вообще ничего здесь не выделил, нехорошо!

In [ ]:
with open('/content/drive/My Drive/Семантические анализаторы/article/Adaskina_IMS_2015_rus.txt', 'r', encoding='utf-8') as file:
    keywords_full = extract_keywords_with_pullenti(file)
keywords_full

[('ю адаскин', 4),
 ('п ребров', 4),
 ('а попов', 3),
 ('в', 2),
 ('адаскин', 1),
 ('адаскина', 1),
 ('ю', 1),
 ('попов', 1),
 ('а', 1),
 ('м', 1),
 ('ребров', 1),
 ('реброва', 1),
 ('п', 1),
 ('университет', 1),
 ('санкт петербургский государственный университет', 1),
 ('спбгу', 1)]